# Center-track ID stability — fixes + parameter sweep

Branch `fix/center-track-id-stability`. Three changes:

1. `find_center_track` is now **stateful** (hysteresis) — the audit's `center_track_id` no longer flickers between two tracks that briefly share the center lane.
2. The audit passes `frame=` to the tracker and enables **ReID** by default → matches the shipping `--preset` pipeline; gallery revival can reclaim an id after an occlusion gap.
3. `TrackStore._maybe_mark_sticky` **latches** the sticky role onto one track instead of overwriting it every frame.

Baseline to beat (current `develop`, `mn1-2.mov`, 713 frames):
`track_id_change_count = 37`, `center_missing_stabilized_count = 61`, `center_missing_raw_count = 78`.
Golden target: `track_id_change_count ≤ 15`.

Needs the confidential `mn1-2.mov`. Runtime > GPU.

In [ ]:
!nvidia-smi -L || echo 'No GPU'

## 1. Install + place `mn1-2.mov`

In [ ]:
import os

REPO = '/content/rf-detr'
BRANCH = 'fix/center-track-id-stability'
if not os.path.exists(REPO):
    !git clone --branch {BRANCH} https://github.com/shingo257/rf-detr.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log --oneline -4
!pip -q install -e '.[reid]' onnx onnxscript
!pip -q install 'pytest>=7.2,<10' 'pytest-xdist>=3.6,<4' 'pytest-timeout>=2,<3'

# mn1-2.mov: upload, or copy from Drive. The audit resolves this path by default.
SRC = 'confidential/media/input/mn1-2.mov'
os.makedirs('confidential/media/input', exist_ok=True)
if not os.path.exists(SRC):
    from google.colab import files  # noqa: I001
    up = files.upload()
    name = next(iter(up))
    if name != SRC:
        os.replace(name, SRC)
assert os.path.exists(SRC), SRC
print('ready:', SRC, os.path.getsize(SRC) // 1024, 'KiB')

## 2. Regression check — tracking + audit tests must stay green

In [ ]:
!python -m pytest -q --timeout=300 \
  tests/rfdetr_demo/test_center_track_selection.py \
  tests/rfdetr_demo/test_person_track_pipeline.py \
  tests/rfdetr_demo/test_tracking_audit.py \
  tests/rfdetr_demo/test_tracking_audit_golden.py \
  tests/rfdetr_demo/test_detection_track.py \
  tests/rfdetr_demo/test_person_track_settings.py

## 3. Sweep

One-at-a-time around the defaults. `run_center_tracking_audit` reads the tracker knobs from env via `person_track_settings_from_env`, so we just set env vars per run. Full 713 frames each; ~2–4 min/run on a T4.

In [ ]:
import os, time, json
from pathlib import Path
from rfdetr_demo.media.audit.tracking import run_center_tracking_audit

SRC = Path('confidential/media/input/mn1-2.mov')
KNOBS = ['RFDETR_STICKY_MAX_MISSED', 'RFDETR_MOTION_GATE_FACTOR',
         'RFDETR_TRACK_REID', 'RFDETR_REID_SIMILARITY', 'RFDETR_REID_GALLERY_FRAMES']
BASE = {
    'RFDETR_STICKY_CENTER_TRACK': '1',
    'RFDETR_STICKY_MAX_MISSED': '4',
    'RFDETR_MOTION_GATE_FACTOR': '1.5',
    'RFDETR_TRACK_REID': '1',
    'RFDETR_REID_SIMILARITY': '0.5',
    'RFDETR_REID_GALLERY_FRAMES': '60',
}

GRID = [
    ('baseline (fixes, defaults)', {}),
    ('sticky_max_missed=8', {'RFDETR_STICKY_MAX_MISSED': '8'}),
    ('sticky_max_missed=12', {'RFDETR_STICKY_MAX_MISSED': '12'}),
    ('motion_gate_factor=1.0', {'RFDETR_MOTION_GATE_FACTOR': '1.0'}),
    ('motion_gate_factor=2.5', {'RFDETR_MOTION_GATE_FACTOR': '2.5'}),
    ('reid_similarity=0.35', {'RFDETR_REID_SIMILARITY': '0.35'}),
    ('reid_similarity=0.65', {'RFDETR_REID_SIMILARITY': '0.65'}),
    ('reid_gallery=120', {'RFDETR_REID_GALLERY_FRAMES': '120'}),
    ('reid OFF (isolate hysteresis)', {'RFDETR_TRACK_REID': '0'}),
]

rows = []
for label, override in GRID:
    for k in KNOBS:
        os.environ.pop(k, None)
    os.environ.update({**BASE, **override})
    t0 = time.time()
    summary = run_center_tracking_audit(
        source_path=SRC,
        sample_interval=100,
        save_on_center_event=False,
    )
    ev = summary.evaluation
    rows.append({
        'config': label,
        'id_changes': ev['track_id_change_count'],
        'missing_stab': ev['center_missing_stabilized_count'],
        'missing_raw': ev['center_missing_raw_count'],
        'ghost_only': ev['center_ghost_only_count'],
        'sec': round(time.time() - t0, 1),
    })
    print(rows[-1])

print('\n=== SWEEP (golden id_changes 31, target <=15; current develop baseline 37 / miss_stab 61 / miss_raw 78) ===')
hdr = f"{'config':32} {'id_ch':>6} {'miss_stab':>10} {'miss_raw':>9} {'ghost':>6} {'sec':>6}"
print(hdr); print('-' * len(hdr))
for r in sorted(rows, key=lambda r: r['id_changes']):
    print(f"{r['config']:32} {r['id_changes']:6} {r['missing_stab']:10} {r['missing_raw']:9} {r['ghost_only']:6} {r['sec']:6}")
json.dump(rows, open('/content/center_track_sweep.json', 'w'), indent=2)

## 4. Best config — full detail + ID-change list

Set `BEST` from the table above, then inspect *which* frames still switch.

In [ ]:
BEST = {}  # e.g. {'RFDETR_STICKY_MAX_MISSED': '8', 'RFDETR_REID_SIMILARITY': '0.35'}

for k in KNOBS:
    os.environ.pop(k, None)
os.environ.update({**BASE, **BEST})
summary = run_center_tracking_audit(source_path=SRC, sample_interval=100, save_on_center_event=False)
ev = summary.evaluation
print(json.dumps({k: ev[k] for k in [
    'verdict', 'track_id_change_count', 'center_missing_stabilized_count',
    'center_missing_raw_count', 'center_ghost_only_count', 'first_center_loss_frame_index',
]}, ensure_ascii=False, indent=2))
print('\nremaining ID changes:')
for c in ev['track_id_changes']:
    print(f"  frame {c['frame_index']:4}  {c['from_track_id']} -> {c['to_track_id']}")